# Unseen University — Database Registry
## Database Methods Practical Examination
### Module Code: 5N0783 | Weighting: 50% | Time: 2 Hours
### Instructor: Joshua Aaron

---

## Instructions

Read all instructions carefully before beginning.

- **Run the Setup Cell first** (Task 0). Every other task depends on it.
- Work through Tasks 1 to 5 in order. Each task builds on the previous one.
- Write your code in the empty cells provided beneath each question.
- If a cell produces an error, read the error message carefully — it usually tells you exactly what needs fixing.
- Save your notebook regularly using Ctrl+S (or Cmd+S on Mac).
- Before submitting, run Kernel > Restart & Run All to ensure all cells execute cleanly.

---

## Scenario

Welcome to Unseen University, Ankh-Morpork's premier institution of magical learning. After centuries of record-keeping by increasingly confused wizards (and one orangutan), the University has decided to modernise its student registry.

You have been appointed as the new Database Registrar. The database contains four tables:

- **teachers** — the University's academic staff, from the Archchancellor to junior wizards
- **programs** — the degree programmes on offer, each led by a senior wizard
- **modules** — individual courses of magical study, each assigned to a teacher
- **students** — enrolled students, each registered to a programme

Notice that no table contains a list. Instead, relationships between tables are handled using **foreign keys** and **queries** — exactly as you discussed in class.

Good luck. The Librarian is watching.

---

## Mark Breakdown

| Task | Description | Marks |
|------|-------------|-------|
| 1 | Database Exploration | 10 |
| 2 | Queries | 15 |
| 3 | Data Entry Form | 10 |
| 4 | Report and Visualisation | 10 |
| 5 | Import External Data | 5 |
| **Total** | | **50** |

---
## Task 0 — Setup (Run This First)

Run the cell below before attempting any other task. It creates the database, builds all four tables, and loads the sample data. You do not need to modify this cell.

In [ ]:
# Uncomment the line below if running in Google Colab
# %pip install ipywidgets matplotlib pandas

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create the database
conn   = sqlite3.connect('unseen_university.db')
cursor = conn.cursor()

# ── Create Tables ─────────────────────────────────────────────────────────────

cursor.execute('''
    CREATE TABLE IF NOT EXISTS teachers (
        teacher_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        name         TEXT    NOT NULL,
        dob          TEXT,
        email        TEXT
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS programs (
        program_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        program_name TEXT    NOT NULL,
        program_lead INTEGER,
        FOREIGN KEY (program_lead) REFERENCES teachers(teacher_id)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS modules (
        module_id    INTEGER PRIMARY KEY AUTOINCREMENT,
        module_name  TEXT    NOT NULL,
        teacher_id   INTEGER,
        duration_hrs INTEGER,
        FOREIGN KEY (teacher_id) REFERENCES teachers(teacher_id)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        student_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        name         TEXT    NOT NULL,
        dob          TEXT,
        gender       TEXT,
        program_id   INTEGER,
        FOREIGN KEY (program_id) REFERENCES programs(program_id)
    )
''')

# ── Seed Data ─────────────────────────────────────────────────────────────────

teachers_data = [
    (1, 'Mustrum Ridcully',   '1940-03-15', 'archchancellor@uu.am'),
    (2, 'Ponder Stibbons',    '1975-06-22', 'p.stibbons@uu.am'),
    (3, 'The Librarian',      '1932-01-01', 'ook@uu.am'),
    (4, 'The Bursar',         '1948-09-10', 'bursar@uu.am'),
    (5, 'Windle Poons',       '1899-11-30', 'w.poons@uu.am'),
    (6, 'Adrian Turnipseed',  '1980-04-17', 'a.turnipseed@uu.am'),
]

programs_data = [
    (1, 'Applied Thaumaturgy',              1),
    (2, 'Practical Necromancy',             5),
    (3, 'Theoretical High Energy Magic',    2),
    (4, 'Mending and Mentalism',            4),
]

modules_data = [
    (1, 'Introduction to Thaumic Flux',        2, 60),
    (2, 'Advanced Fireballing',                1, 90),
    (3, 'Post-Mortem Communications',          5, 120),
    (4, 'Octiron Metallurgy',                  3, 45),
    (5, 'Staff Maintenance and Etiquette',     1, 30),
    (6, 'Applied Conundrumatically Studies',   6, 75),
    (7, 'Defensive Mumblings',                 4, 60),
    (8, 'History of Magic (Abridged)',          3, 90),
]

students_data = [
    (1,  'Eskarina Smith',        '2001-04-12', 'F', 1),
    (2,  'Victor Tugelbend',      '2000-08-03', 'M', 3),
    (3,  'Tiffany Aching',        '2003-01-27', 'F', 4),
    (4,  'Mort Sto Helit',        '1999-11-05', 'M', 2),
    (5,  'Susan Sto Helit',       '2001-07-19', 'F', 3),
    (6,  'Carrot Ironfoundersson', '1998-06-21', 'M', 1),
    (7,  'Agnes Nitt',            '2002-03-08', 'F', 4),
    (8,  'William de Worde',      '2000-12-14', 'M', 3),
    (9,  'Cheery Littlebottom',   '2001-09-30', 'F', 1),
    (10, 'Cosmo Lavish',          '1999-05-17', 'M', 2),
    (11, 'Reaper Man',            '2003-10-31', 'M', 2),
    (12, 'Glenda Sugarbean',      '2000-02-28', 'F', 4),
    (13, 'Juliet Stollop',        '2001-08-11', 'F', 4),
    (14, 'Nutt Goblin',           '2002-04-01', 'M', 3),
    (15, 'Polly Perks',           '2000-11-22', 'F', 1),
    (16, 'Maladicta',             '2001-06-15', 'F', 2),
    (17, 'Otto Chriek',           '1978-01-01', 'M', 2),
    (18, 'Sam Vimes Jr',          '2003-03-17', 'M', 1),
    (19, 'Angua von Uberwald',    '1998-09-04', 'F', 3),
    (20, 'Dorfl Golem',           '2000-07-07', 'M', 1),
    (21, 'Wuffles the Dog',       '2004-05-05', 'M', 4),
    (22, 'Cut-Me-Own-Throat Dibbler', '1997-12-12', 'M', 3),
    (23, 'Magrat Garlick',        '1999-04-20', 'F', 4),
    (24, 'Granny Weatherwax',     '1955-08-01', 'F', 1),
    (25, 'Nanny Ogg',             '1950-02-14', 'F', 4),
]

cursor.executemany('INSERT OR IGNORE INTO teachers VALUES (?,?,?,?)',  teachers_data)
cursor.executemany('INSERT OR IGNORE INTO programs VALUES (?,?,?)',    programs_data)
cursor.executemany('INSERT OR IGNORE INTO modules  VALUES (?,?,?,?)', modules_data)
cursor.executemany('INSERT OR IGNORE INTO students VALUES (?,?,?,?,?)', students_data)
conn.commit()

print('Unseen University database is ready.')
print(f'  Teachers : {pd.read_sql("SELECT COUNT(*) FROM teachers", conn).iloc[0,0]}')
print(f'  Programs : {pd.read_sql("SELECT COUNT(*) FROM programs", conn).iloc[0,0]}')
print(f'  Modules  : {pd.read_sql("SELECT COUNT(*) FROM modules",  conn).iloc[0,0]}')
print(f'  Students : {pd.read_sql("SELECT COUNT(*) FROM students", conn).iloc[0,0]}')

---
## Task 1 — Database Exploration (10 marks)

The Archchancellor has demanded a full audit of the registry before the new term begins.

### Task 1a — View the structure of the students table (3 marks)

Display the column names and data types for the `students` table using `PRAGMA table_info`.

In [ ]:
# Your code here


### Task 1b — Count the enrolled students (3 marks)

Using SQL, find out how many students are currently enrolled at Unseen University.

In [ ]:
# Your code here


### Task 1c — Preview the modules table (4 marks)

Load the `modules` table into a pandas DataFrame and display the first 5 rows.

In [ ]:
# Your code here


---
## Task 2 — Queries (15 marks)

Ponder Stibbons needs several reports generated for the Faculty Meeting.

### Task 2a — SQL: Find long modules (5 marks)

Using SQL, retrieve all modules where `duration_hrs` is greater than 60. Display the `module_name` and `duration_hrs` columns only.

In [ ]:
# Your code here


### Task 2b — pandas: Filter female students (5 marks)

Load the `students` table into a DataFrame, then use pandas boolean indexing to find all students where `gender` is `'F'`.

In [ ]:
# Your code here


### Task 2c — SQL: Sort teachers by name (5 marks)

Using SQL, display all teachers sorted alphabetically by `name` in ascending order.

In [ ]:
# Your code here


---
## Task 3 — Data Entry Form (10 marks)

New students keep arriving at the Registry office (several are noticeably on fire, which Ridcully considers a good sign). Create an ipywidgets form to register a new student.

Your form must include:
- A text input for the student's **name**
- A text input for their **date of birth** (format: YYYY-MM-DD)
- A dropdown for **gender** with options: `'F'`, `'M'`, `'Other'`
- A dropdown for **programme**, populated dynamically from the `programs` table
- A **Submit** button

When submitted, the form should:
1. Insert the new student into the `students` table
2. Display a confirmation message showing the student's name and chosen programme
3. Clear the name and date of birth fields ready for the next entry

**Marking breakdown:**
- Widgets created correctly: 4 marks
- INSERT executes and commits to the database: 3 marks
- Confirmation message displayed: 2 marks
- Form is displayed and functional: 1 mark

In [ ]:
# Hint: load programs from the database to populate the dropdown
# programs_df = pd.read_sql("SELECT program_id, program_name FROM programs", conn)
# program_options = [(row['program_name'], row['program_id']) for _, row in programs_df.iterrows()]

# Your code here


---
## Task 4 — Report and Visualisation (10 marks)

The Bursar has requested a visual summary for the next Senate meeting (he will not read prose under any circumstances).

### Task 4a — Query: Students per programme (4 marks)

Write a SQL query that uses `JOIN` and `GROUP BY` to count how many students are enrolled in each programme. Your result should show the **programme name** and a **count** column.

Hint: you will need to join `students` and `programs` on `program_id`.

In [ ]:
# Your code here


### Task 4b — Chart: Bar chart of students per programme (6 marks)

Using the result from Task 4a, create a bar chart showing the number of students in each programme. Your chart must include a title, axis labels, and `plt.tight_layout()`.

In [ ]:
# Your code here


---
## Task 5 — Import External Data (5 marks)

A batch of late-enrolling students has arrived by carrier pigeon from Überwald. Their records are in the CSV data below.

First, run the cell below to create the CSV file. Then write code to:
1. Read the CSV into a pandas DataFrame
2. Append those records to the `students` table using `to_sql`
3. Verify the import by displaying the total student count after the import

In [ ]:
# Run this cell first — it creates the CSV file of new students
csv_content = """name,dob,gender,program_id
Igor von Uberwald,2002-05-14,M,2
Igorina,2003-09-01,F,4
Count Arthur Notfaroutoe,1987-10-31,M,2
Lacrimosa,2001-03-21,F,3
"""
with open('late_enrolments.csv', 'w') as f:
    f.write(csv_content)
print('late_enrolments.csv created — 4 late arrivals from Überwald.')

In [ ]:
# Your code here


---

## End of Examination

Before submitting:
- Run **Kernel > Restart & Run All** to confirm every cell executes without errors
- Check that all outputs are visible in your notebook
- Save your file one final time with Ctrl+S

| Task | Marks |
|------|-------|
| 1 — Exploration | 10 |
| 2 — Queries | 15 |
| 3 — Data Entry Form | 10 |
| 4 — Visualisation | 10 |
| 5 — Import | 5 |
| **Total** | **50** |

*The Librarian would like it noted that all library books must be returned before results are released. Ook.*

In [ ]:
# Close the database connection when you are finished
conn.close()